## T01

<!-- merged from: T01.ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Construct Deutsch circuit for f(x) = 0 (Constant-0 Oracle)
qc_const0 = QuantumCircuit(2, 1)
# Step 1: Initialize states - input in |+>, ancilla in |->
qc_const0.x(1)
qc_const0.h(0)
qc_const0.h(1)
qc_const0.barrier()
# Step 2: Constant Oracle f(x) = 0 (Identity / no-op on target)
# Ancilla remains unchanged since y ^ 0 = y
qc_const0.barrier()
# Step 3: Interference stage & measurement of query qubit
qc_const0.h(0)
qc_const0.measure(0, 0)
simulator = AerSimulator()
counts_const0 = simulator.run(qc_const0, shots=1000).result().get_counts()
print("--- Deutsch Circuit for f(x) = 0 ---")
print(qc_const0)
print("\nMeasured Counts:", counts_const0)
verdict = "Constant" if '0' in counts_const0 and len(counts_const0) == 1 else "Balanced"
print(f"Function Classification: {verdict} (Observed bit: 0)")

--- Deutsch Circuit for f(x) = 0 ---
     ┌───┐      ░  ░ ┌───┐┌─┐
q_0: ┤ H ├──────░──░─┤ H ├┤M├
     ├───┤┌───┐ ░  ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░──░───────╫─
     └───┘└───┘ ░  ░       ║ 
c: 1/══════════════════════╩═
                           0 

Measured Counts: {'0': 1000}
Function Classification: Constant (Observed bit: 0)


## T02

<!-- merged from: T02.ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

simulator = AerSimulator()

# Construct Deutsch circuit for f(x) = x (Balanced Identity Oracle)
qc_bal_x = QuantumCircuit(2, 1)
# Step 1: State preparation
qc_bal_x.x(1)
qc_bal_x.h(0)
qc_bal_x.h(1)
qc_bal_x.barrier()
# Step 2: Balanced Oracle f(x) = x (CX gate from q0 to q1)
qc_bal_x.cx(0, 1)
qc_bal_x.barrier()
# Step 3: Interference & measurement
qc_bal_x.h(0)
qc_bal_x.measure(0, 0)
counts_bal_x = simulator.run(qc_bal_x, shots=1000).result().get_counts()
print("--- Deutsch Circuit for f(x) = x ---")
print(qc_bal_x)
print("\nMeasured Counts:", counts_bal_x)
verdict = "Balanced" if '1' in counts_bal_x and len(counts_bal_x) == 1 else "Constant"
print(f"Function Classification: {verdict} (Observed bit: 1)")

--- Deutsch Circuit for f(x) = x ---
     ┌───┐      ░       ░ ┌───┐┌─┐
q_0: ┤ H ├──────░───■───░─┤ H ├┤M├
     ├───┤┌───┐ ░ ┌─┴─┐ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤ X ├─░───────╫─
     └───┘└───┘ ░ └───┘ ░       ║ 
c: 1/═══════════════════════════╩═
                                0 

Measured Counts: {'1': 1000}
Function Classification: Balanced (Observed bit: 1)


## T03 (1)

<!-- merged from: T03 (1).ipynb (Jupy Tools) -->

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator


def build_deutsch_oracle(oracle_type):
    oracle_qc = QuantumCircuit(2, name=f"Oracle_{oracle_type}")

    if oracle_type == "constant_0":
        pass  # f(x) = 0

    elif oracle_type == "constant_1":
        oracle_qc.x(1)  # f(x) = 1

    elif oracle_type == "balanced_x":
        oracle_qc.cx(0, 1)  # f(x) = x

    elif oracle_type == "balanced_not_x":
        oracle_qc.x(0)
        oracle_qc.cx(0, 1)  # f(x) = ~x
        oracle_qc.x(0)

    else:
        raise ValueError(f"Unknown oracle type: {oracle_type}")

    return oracle_qc


def run_deutsch_algorithm(oracle_type):
    qc = QuantumCircuit(2, 1)

    # Prepare |0>|1>
    qc.x(1)

    # Create superposition
    qc.h([0, 1])

    qc.barrier()

    # Apply oracle
    qc.compose(
        build_deutsch_oracle(oracle_type),
        inplace=True
    )

    qc.barrier()

    # Interference
    qc.h(0)

    # Measure first qubit
    qc.measure(0, 0)

    # Run simulator
    res = (
        AerSimulator()
        .run(qc, shots=500)
        .result()
        .get_counts()
    )

    measured_bit = list(res.keys())[0]

    inferred_type = (
        "Constant" if measured_bit == "0"
        else "Balanced"
    )

    return measured_bit, inferred_type


# Test all four functions
all_types = [
    "constant_0",
    "constant_1",
    "balanced_x",
    "balanced_not_x"
]


print(
    f"{'Function Type':<18}"
    f"{'Measured Bit':<15}"
    f"{'Inferred Class':<15}"
    f"{'Status':<10}"
)

print("=" * 58)


for fn in all_types:
    bit, outcome = run_deutsch_algorithm(fn)

    expected = (
        "Constant"
        if "constant" in fn
        else "Balanced"
    )

    match = "PASS" if outcome == expected else "FAIL"

    print(
        f"{fn:<18}"
        f"{bit:<15}"
        f"{outcome:<15}"
        f"{match:<10}"
    )

Function Type     Measured Bit   Inferred Class Status    
constant_0        0              Constant       PASS      
constant_1        0              Constant       PASS      
balanced_x        1              Balanced       PASS      
balanced_not_x    1              Balanced       PASS      


## T04 (1)

<!-- merged from: T04 (1).ipynb (Jupy Tools) -->

In [ ]:
# Conceptual mapping of phase kickback to binary classification
# Feature vector x is evaluated by decision boundary f(x) in {0, 1}
# Class 0: Negative / Benign (Constant parity across baseline)
# Class 1: Positive / Malicious (Boundary flip via phase kickback)
desc = [
 "--- Binary Classification via Quantum Phase Encoding ---",
 "",
 "[Feature State |x>] ---> [ H ] ----o---- [ H ] ---> [ Readout: Class Label ]",
 " |",
 "[Classifier Ancilla] -> [ X, H ] -[U_f] -------- (Phase Kickback: (-1)^f(x))",
 "",
 "Step 1: Superposition encodes uniform prior over feature combinations.",
 "Step 2: Oracle U_f acts as hypothesis evaluator: |x>|-> -> (-1)^f(x) |x>|->.",
 "Step 3: Global phase alignment determines class boundary:",
 " * Null-shift: Output 0 => Class 0 (Homogeneous / Constant parity).",
 " * Phase-inversion: Output 1 => Class 1 (Separable / Balanced decision boundary)."
]
print("\n".join(desc))

--- Binary Classification via Quantum Phase Encoding ---

[Feature State |x>] ---> [ H ] ----o---- [ H ] ---> [ Readout: Class Label ]
 |
[Classifier Ancilla] -> [ X, H ] -[U_f] -------- (Phase Kickback: (-1)^f(x))

Step 1: Superposition encodes uniform prior over feature combinations.
Step 2: Oracle U_f acts as hypothesis evaluator: |x>|-> -> (-1)^f(x) |x>|->.
Step 3: Global phase alignment determines class boundary:
 * Null-shift: Output 0 => Class 0 (Homogeneous / Constant parity).
 * Phase-inversion: Output 1 => Class 1 (Separable / Balanced decision boundary).


## T05 (1)

<!-- merged from: T05 (1).ipynb (Jupy Tools) -->

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Introduce deliberate rotation error delta on the Hadamard gate: Ry(pi/2 + delta)
def imperfect_hadamard(qc, qubit, delta_theta):
 qc.ry(np.pi / 2 + delta_theta, qubit)
delta_errors = [0.0, 0.1, 0.25, 0.5, 0.75]
shots = 2000
sim = AerSimulator()
print("--- Imperfect Hadamard Error Sensitivity (Balanced f(x) = x) ---")
print(f"{'Error Delta (rad)':<20}{'P(1) [Correct]':<18}{'P(0) [Leakage]':<18}{'Fidelity'}")
print("=" * 68)
fidelity_results = []
for delta in delta_errors:
 qc_noisy = QuantumCircuit(2, 1)
 qc_noisy.x(1)
 qc_noisy.h(1)
 # Apply imperfect Hadamard on query qubit
 imperfect_hadamard(qc_noisy, 0, delta)
 # Balanced oracle
 qc_noisy.cx(0, 1)
 # Final imperfect Hadamard
 imperfect_hadamard(qc_noisy, 0, delta)
 qc_noisy.measure(0, 0)
 counts = sim.run(qc_noisy, shots=shots).result().get_counts()
 p1 = counts.get('1', 0) / shots
 p0 = counts.get('0', 0) / shots
 fidelity_results.append(p1)
 print(f"{delta:<20.2f}{p1:<18.4f}{p0:<18.4f}{p1*100:.1f}%")

--- Imperfect Hadamard Error Sensitivity (Balanced f(x) = x) ---
Error Delta (rad)   P(1) [Correct]    P(0) [Leakage]    Fidelity
0.00                0.0000            1.0000            0.0%
0.10                0.0000            1.0000            0.0%
0.25                0.0000            1.0000            0.0%
0.50                0.0000            1.0000            0.0%
0.75                0.0000            1.0000            0.0%
